In [5]:
# !pip install -q transformer-lens transformers accelerate

from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("Qwen/Qwen3-0.6B")

print("Model loaded!")
print(f"Layers: {model.cfg.n_layers}")
print(f"Hidden dimension: {model.cfg.d_model}")

/tmp/ipykernel_773/2080072307.py:5: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained("Qwen/Qwen3-0.6B")


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen3-0.6B into HookedTransformer
Model loaded!
Layers: 28
Hidden dimension: 1024


In [10]:
prompt = """Solve this problem step by step.

Alice is older than Bob.
Bob is older than Charlie.

Question: Who is the oldest?

Explain your reasoning and then give the final answer."""

tokens = model.to_tokens(prompt)

logits, cache = model.run_with_cache(tokens)

print("Number of tokens:", tokens.shape[1])
print("Cached activation keys:", len(cache))

Number of tokens: 38
Cached activation keys: 677


In [11]:
for layer in [0, 7, 14, 21, 27]:
    activation = cache[f"blocks.{layer}.hook_resid_post"]
    print(
        f"Layer {layer}:",
        activation.shape
    )

Layer 0: torch.Size([1, 38, 1024])
Layer 7: torch.Size([1, 38, 1024])
Layer 14: torch.Size([1, 38, 1024])
Layer 21: torch.Size([1, 38, 1024])
Layer 27: torch.Size([1, 38, 1024])


In [4]:
clean_prompt = """Alice is older than Bob.
Bob is older than Charlie.
Who is the oldest?"""

corrupt_prompt = """Alice is younger than Bob.
Bob is older than Charlie.
Who is the oldest?"""

clean_tokens = model.to_tokens(clean_prompt)
corrupt_tokens = model.to_tokens(corrupt_prompt)

print("Clean:")
print(model.to_str_tokens(clean_tokens))

print("\nCorrupted:")
print(model.to_str_tokens(corrupt_tokens))

Clean:
['Alice', ' is', ' older', ' than', ' Bob', '.\n', 'Bob', ' is', ' older', ' than', ' Charlie', '.\n', 'Who', ' is', ' the', ' oldest', '?']

Corrupted:
['Alice', ' is', ' younger', ' than', ' Bob', '.\n', 'Bob', ' is', ' older', ' than', ' Charlie', '.\n', 'Who', ' is', ' the', ' oldest', '?']


In [5]:
clean_logits = model(clean_tokens)
corrupt_logits = model(corrupt_tokens)

clean_next = clean_logits[0, -1].argmax()
corrupt_next = corrupt_logits[0, -1].argmax()

print("Clean prediction:", repr(model.to_string(clean_next)))
print("Corrupted prediction:", repr(model.to_string(corrupt_next)))

Clean prediction: ' A'
Corrupted prediction: ' A'


In [6]:
clean_prompt = """Answer the question with exactly one name.

Alice is older than Bob.
Bob is older than Charlie.

Who is the oldest?
Answer: Alice"""

corrupt_prompt = """Answer the question with exactly one name.

Alice is younger than Bob.
Bob is older than Charlie.

Who is the oldest?
Answer: Bob"""

clean_tokens = model.to_tokens(clean_prompt)
corrupt_tokens = model.to_tokens(corrupt_prompt)

print("Clean:")
print(model.to_str_tokens(clean_tokens))

print("\nCorrupted:")
print(model.to_str_tokens(corrupt_tokens))

Clean:
['Answer', ' the', ' question', ' with', ' exactly', ' one', ' name', '.\n\n', 'Alice', ' is', ' older', ' than', ' Bob', '.\n', 'Bob', ' is', ' older', ' than', ' Charlie', '.\n\n', 'Who', ' is', ' the', ' oldest', '?\n', 'Answer', ':', ' Alice']

Corrupted:
['Answer', ' the', ' question', ' with', ' exactly', ' one', ' name', '.\n\n', 'Alice', ' is', ' younger', ' than', ' Bob', '.\n', 'Bob', ' is', ' older', ' than', ' Charlie', '.\n\n', 'Who', ' is', ' the', ' oldest', '?\n', 'Answer', ':', ' Bob']


In [10]:
clean_prompt = """Answer with exactly one name.

Alice is older than Bob.
Bob is older than Charlie.

Who is the oldest?
Answer:"""

corrupt_prompt = """Answer with exactly one name.

Alice is younger than Bob.
Bob is older than Charlie.

Who is the oldest?
Answer:"""

clean_tokens = model.to_tokens(clean_prompt)
corrupt_tokens = model.to_tokens(corrupt_prompt)

clean_logits = model(clean_tokens)
corrupt_logits = model(corrupt_tokens)

# Probability of the two candidate answers
for name in [" Alice", " Bob"]:
    token = model.to_single_token(name)
    clean_prob = clean_logits[0, -1].softmax(-1)[token].item()
    corrupt_prob = corrupt_logits[0, -1].softmax(-1)[token].item()

    print(f"{name}: clean={clean_prob:.4f}, corrupted={corrupt_prob:.4f}")

 Alice: clean=0.1046, corrupted=0.0604
 Bob: clean=0.0052, corrupted=0.0048


## Outcome

The initial toy age-ordering task was not suitable for activation patching with Qwen3-0.6B: the model did not realiably distinguish the clean and corrupted conditions. We therefore do not interpret any activation patching results from this task.

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
chat_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

messages = [
    {
        "role": "user",
        "content": """Alice is older than Bob.
Bob is older than Charlie.

Who is the oldest?
Explain your reasoning and give the final answer."""
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

inputs = tokenizer(text, return_tensors="pt").to(chat_model.device)

outputs = chat_model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.6,
    top_p=0.95,
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)

print(response)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

<think>
Okay, let's see. The problem says Alice is older than Bob, and Bob is older than Charlie. The question is asking who is the oldest. Hmm, so I need to figure out the hierarchy here.

First, let's break down the information given. Alice is older than Bob. That means Alice's age is higher than Bob's. Then Bob is older than Charlie. So Bob's age is higher than Charlie's. Now, if both of these statements are true, how does that affect the order of ages?

Let me visualize this. If Alice is older than Bob, then Alice comes first. Then Bob is older than Charlie, so Charlie comes after Bob. So putting it all together, the order from youngest to oldest would be Charlie, then Bob, then Alice. Wait, but does that make sense?

Wait, hold on. Let me think again. If Alice is older than Bob, then Alice is older than Charlie? No, because Charlie is only mentioned in the second statement. The first statement says Alice is older than Bob, but does that mean Alice is older than Charlie? The proble

## Qwen3 Chat-Template Sanity Check

The initial raw-text interface produced unreliable behavior on the toy
reasoning task. We therefore tested Qwen3-0.6B using its official chat
template with thinking mode enabled.

The chat-formatted model produced a coherent chain-of-thought trajectory
for the age-ordering task, suggesting that the earlier failure was at least
partly due to the inference interface rather than the model's inability to
solve the task.

Next: determine whether the same chat-formatted model can be connected to
our activation-inspection pipeline.

In [2]:
# Inspect the exact chat-formatted prompt we gave Qwen
print(text)

<|im_start|>user
Alice is older than Bob.
Bob is older than Charlie.

Who is the oldest?
Explain your reasoning and give the final answer.<|im_end|>
<|im_start|>assistant



In [6]:
# Tokenize the chat-formatted prompt with TransformerLens
chat_tokens = model.to_tokens(text)

print("Number of tokens:", chat_tokens.shape[1])
print(model.to_str_tokens(chat_tokens))

Number of tokens: 35
['<|im_start|>', 'user', '\n', 'Alice', ' is', ' older', ' than', ' Bob', '.\n', 'Bob', ' is', ' older', ' than', ' Charlie', '.\n\n', 'Who', ' is', ' the', ' oldest', '?\n', 'Ex', 'plain', ' your', ' reasoning', ' and', ' give', ' the', ' final', ' answer', '.', '<|im_end|>', '\n', '<|im_start|>', 'assistant', '\n']


In [11]:
generated = model.generate(
    chat_tokens,
    max_new_tokens=500,
    temperature=0.6,
    top_p=0.95,
    verbose=False,
)

print(model.to_string(generated))

["<|im_start|>user\nAlice is older than Bob.\nBob is older than Charlie.\n\nWho is the oldest?\nExplain your reasoning and give the final answer.<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, let's see. The problem says Alice is older than Bob, and Bob is older than Charlie. I need to figure out who is the oldest among them. Hmm.\n\nSo, first, let's break down the information. Alice > Bob and Bob > Charlie. So starting from the top, Alice is the oldest, then Bob, then Charlie. So if we list them in order, it would be Alice, Bob, Charlie. That makes sense. But wait, maybe there's a trick here?\n\nWait, the question is about who is the oldest. The statements are comparing Alice to Bob and Bob to Charlie. So each of these comparisons is a direct comparison. So Alice is older than Bob, and Bob is older than Charlie. Therefore, the order from oldest to youngest is Alice, Bob, Charlie. So the answer should be Alice.\n\nBut let me double-check. Suppose Alice is 20, Bob is 15, and Charlie i

In [12]:
generated_for_cache = generated.clone().detach()

logits, cot_cache = model.run_with_cache(generated_for_cache)

print("Generated sequence tokens:", generated_for_cache.shape[1])
print("Cached activation tensors:", len(cot_cache))

Generated sequence tokens: 350
Cached activation tensors: 677


In [13]:
tokens = model.to_str_tokens(generated_for_cache)

for i, token in enumerate(tokens):
    print(f"{i:3d}: {repr(token)}")

  0: '<|im_start|>'
  1: 'user'
  2: '\n'
  3: 'Alice'
  4: ' is'
  5: ' older'
  6: ' than'
  7: ' Bob'
  8: '.\n'
  9: 'Bob'
 10: ' is'
 11: ' older'
 12: ' than'
 13: ' Charlie'
 14: '.\n\n'
 15: 'Who'
 16: ' is'
 17: ' the'
 18: ' oldest'
 19: '?\n'
 20: 'Ex'
 21: 'plain'
 22: ' your'
 23: ' reasoning'
 24: ' and'
 25: ' give'
 26: ' the'
 27: ' final'
 28: ' answer'
 29: '.'
 30: '<|im_end|>'
 31: '\n'
 32: '<|im_start|>'
 33: 'assistant'
 34: '\n'
 35: '<think>'
 36: '\n'
 37: 'Okay'
 38: ','
 39: ' let'
 40: "'s"
 41: ' see'
 42: '.'
 43: ' The'
 44: ' problem'
 45: ' says'
 46: ' Alice'
 47: ' is'
 48: ' older'
 49: ' than'
 50: ' Bob'
 51: ','
 52: ' and'
 53: ' Bob'
 54: ' is'
 55: ' older'
 56: ' than'
 57: ' Charlie'
 58: '.'
 59: ' I'
 60: ' need'
 61: ' to'
 62: ' figure'
 63: ' out'
 64: ' who'
 65: ' is'
 66: ' the'
 67: ' oldest'
 68: ' among'
 69: ' them'
 70: '.'
 71: ' Hmm'
 72: '.\n\n'
 73: 'So'
 74: ','
 75: ' first'
 76: ','
 77: ' let'
 78: "'s"
 79: ' break'
 8

In [14]:
# Test whether the corrupted prompt reliably changes the model's answer

corrupt_messages = [
    {
        "role": "user",
        "content": """Alice is younger than Bob.
Bob is older than Charlie.

Who is the oldest?
Explain your reasoning and give the final answer."""
    }
]

corrupt_text = tokenizer.apply_chat_template(
    corrupt_messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

corrupt_inputs = tokenizer(
    corrupt_text,
    return_tensors="pt"
).to(chat_model.device)

n_samples = 10

for i in range(n_samples):
    corrupt_output = chat_model.generate(
        **corrupt_inputs,
        max_new_tokens=500,
        temperature=0.6,
        top_p=0.95,
    )

    response = tokenizer.decode(
        corrupt_output[0][corrupt_inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )

    print(f"\n{'='*80}")
    print(f"SAMPLE {i+1}")
    print(f"{'='*80}")
    print(response)


SAMPLE 1
<think>
Okay, let's see. The problem says Alice is younger than Bob, and Bob is older than Charlie. I need to figure out who is the oldest. Hmm.

So, first, let's break down the given information. Alice is younger than Bob. That means Alice's age is less than Bob's. So Bob is older than Alice. Then, Bob is older than Charlie. So Charlie's age is less than Bob's age. Therefore, Charlie is younger than Bob. 

Wait, so if Alice is younger than Bob, and Bob is older than Charlie, then Charlie is younger than Bob. But the question is asking who is the oldest. So between Bob and Charlie, Bob is older. But wait, is there any other possibility? Maybe Alice is older than Charlie? Wait, no, the problem says Alice is younger than Bob. So Alice < Bob. And Bob > Charlie. So Charlie < Bob. So Charlie is the youngest. Bob is the oldest. Therefore, the answer should be Bob. 

Let me check again. If Alice < Bob, then Bob is older than Alice. Bob is older than Charlie, so Charlie is older than

## Corrupted Prompt Reliability Check

We tested the corrupted version of the toy task across 10 independent
samples using temperature 0.6 and top-p 0.95.

The corrupted prompt changed the first relationship from:

Alice > Bob

to:

Alice < Bob

while keeping:

Bob > Charlie

All 10 samples concluded that Bob is the oldest.

Thus, for this toy task, the intervention produces a reliable behavioral
contrast between the clean and corrupted conditions.

## Matched Clean and Corrupted CoT Traces

We now generate one clean and one corrupted reasoning trajectory using the
same model, chat template, sampling parameters, and task structure.

Clean:
Alice is older than Bob.
Bob is older than Charlie.

Corrupted:
Alice is younger than Bob.
Bob is older than Charlie.

The goal is to obtain trajectories that differ in their reasoning while
keeping the task otherwise identical. These paired traces will later be
used for causal activation comparisons.

In [15]:
# Generate one matched clean/corrupted pair

clean_messages = [
    {
        "role": "user",
        "content": """Alice is older than Bob.
Bob is older than Charlie.

Who is the oldest?
Explain your reasoning and give the final answer."""
    }
]

corrupt_messages = [
    {
        "role": "user",
        "content": """Alice is younger than Bob.
Bob is older than Charlie.

Who is the oldest?
Explain your reasoning and give the final answer."""
    }
]

clean_text = tokenizer.apply_chat_template(
    clean_messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

corrupt_text = tokenizer.apply_chat_template(
    corrupt_messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

clean_inputs = tokenizer(
    clean_text,
    return_tensors="pt"
).to(chat_model.device)

corrupt_inputs = tokenizer(
    corrupt_text,
    return_tensors="pt"
).to(chat_model.device)

clean_output = chat_model.generate(
    **clean_inputs,
    max_new_tokens=500,
    temperature=0.6,
    top_p=0.95,
)

corrupt_output = chat_model.generate(
    **corrupt_inputs,
    max_new_tokens=500,
    temperature=0.6,
    top_p=0.95,
)

clean_response = tokenizer.decode(
    clean_output[0][clean_inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)

corrupt_response = tokenizer.decode(
    corrupt_output[0][corrupt_inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)

print("=" * 80)
print("CLEAN TRACE")
print("=" * 80)
print(clean_response)

print("\n" + "=" * 80)
print("CORRUPTED TRACE")
print("=" * 80)
print(corrupt_response)

CLEAN TRACE
<think>
Okay, let's see. The problem says Alice is older than Bob, and Bob is older than Charlie. I need to figure out who is the oldest among them. Hmm, so the question is about determining the order of ages based on these statements.

First, let me break down the information. Alice > Bob, and Bob > Charlie. So starting from the top, Alice is older than Bob, and Bob is older than Charlie. So if I imagine a sequence, it would be Alice, then Bob, then Charlie. Therefore, Alice is the oldest. 

Wait, is there any possibility that someone else could be older? The statements only compare Alice to Bob and Bob to Charlie, but there's no information given about the ages of anyone else. So I think the answer is Alice. 

But let me check again. Suppose Alice is older than Bob, and Bob is older than Charlie. So Alice's age is higher than Bob's, which is higher than Charlie's. So Alice is the oldest. Yeah, that makes sense. There's no contradiction here. So the answer should be Alice.

In [16]:
# Inspect token-level alignment between the clean and corrupted CoTs

clean_think = clean_response.split("<think>", 1)[1].split("</think>", 1)[0].strip()
corrupt_think = corrupt_response.split("<think>", 1)[1].split("</think>", 1)[0].strip()

clean_cot_ids = tokenizer(
    clean_think,
    add_special_tokens=False
)["input_ids"]

corrupt_cot_ids = tokenizer(
    corrupt_think,
    add_special_tokens=False
)["input_ids"]

clean_tokens = tokenizer.convert_ids_to_tokens(clean_cot_ids)
corrupt_tokens = tokenizer.convert_ids_to_tokens(corrupt_cot_ids)

print("=" * 100)
print(f"CLEAN CoT:     {len(clean_tokens)} tokens")
print(f"CORRUPTED CoT: {len(corrupt_tokens)} tokens")
print("=" * 100)

max_len = max(len(clean_tokens), len(corrupt_tokens))

print(f"{'POS':>4} | {'CLEAN':<30} | {'CORRUPTED':<30} | MATCH")
print("-" * 100)

for i in range(max_len):
    clean_tok = clean_tokens[i] if i < len(clean_tokens) else "<NONE>"
    corrupt_tok = corrupt_tokens[i] if i < len(corrupt_tokens) else "<NONE>"
    match = clean_tok == corrupt_tok

    print(
        f"{i:4d} | "
        f"{clean_tok:<30} | "
        f"{corrupt_tok:<30} | "
        f"{match}"
    )

CLEAN CoT:     223 tokens
CORRUPTED CoT: 275 tokens
 POS | CLEAN                          | CORRUPTED                      | MATCH
----------------------------------------------------------------------------------------------------
   0 | Okay                           | Okay                           | True
   1 | ,                              | ,                              | True
   2 | Ġlet                           | Ġlet                           | True
   3 | 's                             | 's                             | True
   4 | Ġsee                           | Ġsee                           | True
   5 | .                              | .                              | True
   6 | ĠThe                           | ĠThe                           | True
   7 | Ġproblem                       | Ġproblem                       | True
   8 | Ġsays                          | Ġsays                          | True
   9 | ĠAlice                         | Ġthat                     

## Controlled Matched Reasoning Traces

The free-running clean and corrupted CoTs were behaviorally successful but
poorly aligned at the token level.

We now construct a more controlled pair using the same reasoning structure.
The clean and corrupted prompts differ only in the underlying age relation,
while the model is asked to follow the same explicit reasoning format.

The purpose is to obtain trajectories whose corresponding reasoning positions
are more directly comparable before performing activation analysis.

In [17]:
# Generate controlled clean/corrupted reasoning traces

controlled_clean_messages = [
    {
        "role": "user",
        "content": """Alice is older than Bob.
Bob is older than Charlie.

Use exactly this reasoning structure:

1. State the relationship between Alice and Bob.
2. State the relationship between Bob and Charlie.
3. Combine the two relationships to determine the oldest person.
4. Give the final answer.

Who is the oldest?"""
    }
]

controlled_corrupt_messages = [
    {
        "role": "user",
        "content": """Alice is younger than Bob.
Bob is older than Charlie.

Use exactly this reasoning structure:

1. State the relationship between Alice and Bob.
2. State the relationship between Bob and Charlie.
3. Combine the two relationships to determine the oldest person.
4. Give the final answer.

Who is the oldest?"""
    }
]

controlled_clean_text = tokenizer.apply_chat_template(
    controlled_clean_messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

controlled_corrupt_text = tokenizer.apply_chat_template(
    controlled_corrupt_messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

controlled_clean_inputs = tokenizer(
    controlled_clean_text,
    return_tensors="pt"
).to(chat_model.device)

controlled_corrupt_inputs = tokenizer(
    controlled_corrupt_text,
    return_tensors="pt"
).to(chat_model.device)

controlled_clean_output = chat_model.generate(
    **controlled_clean_inputs,
    max_new_tokens=300,
    temperature=0.6,
    top_p=0.95,
)

controlled_corrupt_output = chat_model.generate(
    **controlled_corrupt_inputs,
    max_new_tokens=300,
    temperature=0.6,
    top_p=0.95,
)

controlled_clean_response = tokenizer.decode(
    controlled_clean_output[0][controlled_clean_inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)

controlled_corrupt_response = tokenizer.decode(
    controlled_corrupt_output[0][controlled_corrupt_inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)

print("=" * 80)
print("CONTROLLED CLEAN TRACE")
print("=" * 80)
print(controlled_clean_response)

print("\n" + "=" * 80)
print("CONTROLLED CORRUPTED TRACE")
print("=" * 80)
print(controlled_corrupt_response)

CONTROLLED CLEAN TRACE
<think>
Okay, let's see. The problem says Alice is older than Bob, and Bob is older than Charlie. I need to figure out who the oldest is. Let me break it down step by step.

First, from the given information, we know two relationships: Alice > Bob and Bob > Charlie. So, if Alice is older than Bob, and Bob is older than Charlie, then starting from the top, Alice would be older than Charlie. That makes sense because each subsequent person is younger than the previous one. So combining those two relationships, the order should be Alice > Bob > Charlie. Therefore, the oldest person is Alice. 

Wait, but I should make sure there's no other possibility. Since there are only three people mentioned here, and each relationship is a direct comparison, there's no ambiguity. So combining them, the answer should be Alice.
</think>

1. State the relationship between Alice and Bob: Alice is older than Bob.  
2. State the relationship between Bob and Charlie: Bob is older than C

## Conclusions from the sanity-check phase

1. Qwen3-0.6B produces coherent reasoning traces when run with its chat
   template and thinking mode.

2. TransformerLens can process the same chat-formatted input and cache
   activations across the generated reasoning trajectory.

3. The clean/corrupted age-relation task gives a reliable behavioral contrast:
   the clean condition produces Alice as the oldest, while the corrupted
   condition produced Bob in 10/10 sampled runs.

4. Free-running clean and corrupted CoTs are not token-aligned, so naive
   position-by-position activation patching between complete CoTs would not
   have a clear interpretation.

5. Prompting the model to use a fixed reasoning structure improves qualitative
   comparability but does not guarantee logically identical reasoning structure.
   For example, the corrupted trace inferred an ordering between Alice and
   Charlie that was not entailed by the premises.

Therefore, the next phase will separate:
(a) causal localization of the changed input fact, and
(b) causal analysis of semantically matched reasoning steps.